# Feature Engineering para el dataset
En este notebbok construimos los features que necesitamos a partir de la columna URL
* Cantidad de dígitos
* Cantidad de símbolos raros (-, _, @, %, =)
* Presencia de IP en vez de dominio
* Número de parámetros
* Longitud del path
* Entropía del string


In [3]:
import ipaddress
import numpy as np
import pandas as pd
import math
from collections import Counter
from pathlib import Path
from urllib.parse import parse_qsl, urlparse

In [4]:
DATA_PATH = Path('../data/PhiUSIIL_Phishing_URL_Dataset.csv')

df = pd.read_csv(DATA_PATH)

df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [9]:
RARE_SYMBOLS = set('-_@%=')

def count_digits(url:str)-> int:

    s = '' if pd.isna(url) else str(url)

    return sum(ch.isdigit() for ch in s)


def count_rare_symbols(url:str)-> int:

    s = '' if pd.isna(url) else str(url)

    return sum(ch in RARE_SYMBOLS for ch in s)


def has_ip_in_domain(url:str)-> int:

    s = '' if pd.isna(url) else str(url).strip()

    if not s:
        return 0
    
    parsed = urlparse(s if "://" in s else  f'http://{s}')

    host = parsed.hostname

    if not host:
        return 0
    
    try:
        ipaddress.ip_address(host)
        return 1
    except ValueError:
        return 0
    

def count_query_params(url:str)-> int:

    s = '' if pd.isna(url) else str(url).strip()

    if not s:
        return 0
    
    parsed = urlparse(s if "://" in s else f'http://{s}')

    return len(parse_qsl(parsed.query,keep_blank_values=True))


def path_length(url:str)-> int:
    s = '' if pd.isna(url) else str(url).strip()

    if not s:
        return 0
    
    parsed = urlparse(s if "://" in s else f'http://{s}')

    return len(parsed.path or '')


def shannon_entropy(text:str)-> float:

    s = '' if pd.isna(text) else str(text)

    if not s:
        return 0.0
    
    counts = Counter(s)
    total = len(s)

    return -sum((freq / total) * math.log2(freq / total) for freq in counts.values())

In [11]:
URL_COL = 'URL'

df['feat_digit_count'] = df[URL_COL].apply(count_digits)
df['feat_rare_symbol_count'] = df[URL_COL].apply(count_rare_symbols)
df['feat_has_ip_domain'] = df[URL_COL].apply(has_ip_in_domain)
df['feat_num_params'] = df[URL_COL].apply(count_query_params)
df['feat_path_length'] = df[URL_COL].apply(path_length)
df['feat_url_entropy'] = df[URL_COL].apply(shannon_entropy)

new_features = [
    'feat_digit_count',
    'feat_rare_symbol_count',
    'feat_has_ip_domain',
    'feat_num_params',
    'feat_path_length',
    'feat_url_entropy'
]

df[[URL_COL] + new_features].head(20)

,URL,feat_digit_count,feat_rare_symbol_count,feat_has_ip_domain,feat_num_params,feat_path_length,feat_url_entropy
0,https://www.southbankmosaics.com,0,0,0,0,0,3.929229
1,https://www.uni-mainz.de,0,1,0,0,0,3.970176
2,https://www.voicefmradio.co.uk,0,0,0,0,0,4.164735
3,https://www.sfnmjournal.com,0,0,0,0,0,4.060262
4,https://www.rewildingargentina.org,0,0,0,0,0,3.917626
5,https://www.globalreporting.org,0,0,0,0,0,3.929214
6,https://www.saffronart.com,0,0,0,0,0,3.796218
7,https://www.nerdscandy.com,0,0,0,0,0,3.979098
8,https://www.hyderabadonline.in,0,0,0,0,0,4.056565
9,https://www.aap.org,0,0,0,0,0,3.471354


In [12]:
output_path = Path('../data/PhiUSIIL_Phishing_URL_Dataset_feature_engineered.csv')
df.to_csv(output_path, index=False)


print(f'Dataset guardado en: {output_path.resolve()}')
print(df[new_features].describe(include='all'))

Dataset guardado en: C:\Users\Carlos\OneDrive\Documentos\URLsPhising\data\PhiUSIIL_Phishing_URL_Dataset_feature_engineered.csv
       feat_digit_count  feat_rare_symbol_count  feat_has_ip_domain  \
count     235795.000000           235795.000000       235795.000000   
mean           1.886172                0.480875            0.002579   
std           11.893102                3.310744            0.050714   
min            0.000000                0.000000            0.000000   
25%            0.000000                0.000000            0.000000   
50%            0.000000                0.000000            0.000000   
75%            0.000000                0.000000            0.000000   
max         2011.000000              645.000000            1.000000   

       feat_num_params  feat_path_length  feat_url_entropy  
count    235795.000000     235795.000000     235795.000000  
mean          0.060531          3.735325          3.960896  
std           0.553977         23.501818          